## **Generators**

A **generator** is a specialized type of function in Python that allows you to loop over a sequence of data **lazily** (one item at a time) instead of computing and storing the entire dataset in memory all at once.

Standard functions use the `return` keyword to output a final result and instantly terminate, completely destroying their local state. Generators use the **`yield`** keyword instead, which temporarily pauses the function's execution, hands a value back to the caller, and retains its exact state to resume precisely where it left off on the next request.

---

- **🏎️ The Core Problem: Memory Exhaustion**
    - Imagine you need to process a dataset containing 10 million integers. A traditional approach would calculate the entire list and load it into your system's RAM:

```python
# ❌ Dangerous: Creates 10 million integers in memory simultaneously
def get_massive_range(n):
    result = []
    for i in range(n):
        result.append(i)
    return result  # Returns a massive list

data = get_massive_range(10_000_000)  # Consumes hundreds of megabytes of RAM
```

> If you run this with a significantly higher limit, your system will run out of memory and crash (`MemoryError`).

---

- **💡 The Solution: Lazy Evaluation with `yield`**
    - A generator converts this memory-heavy pipeline into an iterator stream. It only computes a number when the loop explicitly demands it, dropping it from memory immediately afterward:

```python
# ⚡ Memory Efficient: Streams numbers one by one on demand
def get_massive_range_generator(n):
    for i in range(n):
        yield i  # Pauses execution and hands 'i' back safely

# Consumes practically ZERO memory initial setup!
data_stream = get_massive_range_generator(10_000_000)

for number in data_stream:
    if number > 3:
        break
    print(number)
# Output: 0, 1, 2, 3
```

---

- **🛠️ The Mechanics Under the Hood: `next()`**
    - When you call a generator function, it **does not execute any code inside the function body.** Instead, it creates and returns a **generator object**. To get values out of it, Python utilizes the `next()` function behind the scenes:

```python
def simple_steps():
    print("🎬 Step 1 starting")
    yield "A"
    print("🔄 Step 2 starting")
    yield "B"

# 1. Instantiate the generator object
gen = simple_steps()

# 2. Trigger execution up to the first yield
print(next(gen))  
# Output:
# 🎬 Step 1 starting
# A

# 3. Resume from step 1 up to the second yield
print(next(gen))  
# Output:
# 🔄 Step 2 starting
# B

# 4. Triggering next() again throws a StopIteration exception
# next(gen) -> ❌ Raises StopIteration (This tells loops when to stop)
```

---

- **🏗️ Method 2: Generator Expressions**
    - Just like Python features list comprehensions, it also offers **generator expressions**. They share an almost identical syntax, but utilize **parentheses `()**` instead of square brackets `[]`.

```python
# ❌ List Comprehension: Creates the entire list in memory right now
heavy_list = [x ** 2 for x in range(1000000)]

# ⚡ Generator Expression: Ready to calculate squares on the fly, using zero RAM
lazy_stream = (x ** 2 for x in range(1000000))

print(next(lazy_stream))  # Output: 0
print(next(lazy_stream))  # Output: 1
```

---

- **🚀 Advanced Feature: Two-Way Communication (`send()`)**
    - Generators aren't just for pulling data out; you can also push data back *into* them while they are paused using the **`.send()`** method. When you use `.send(value)`, the `yield` statement inside the generator receives that value as its assignment payload:

```python
def interactive_counter():
    count = 0
    while True:
        # Pauses here, outputs count. If a value is sent in, it overrides 'jump'
        jump = yield count
        if jump is not None:
            count += jump
        else:
            count += 1

counter = interactive_counter()

print(next(counter))      # Output: 0 (Initializes the generator)
print(next(counter))      # Output: 1
print(counter.send(10))   # Output: 11 (Jumps forward by 10!)
print(next(counter))      # Output: 12
```

---

- **⚖️ Summary Comparison: Lists vs. Generators**

| Feature | List / Collection | Generator |
| --- | --- | --- |
| **Evaluation Strategy** | Eager (Everything processed upfront) | Lazy (Processed on request) |
| **Memory Footprint** | Large (Scales linearly with data size) | Extremely Small (Constant space complexity) |
| **Access Patterns** | Can read indexes arbitrarily (`data[5]`) | Sequential access only via `next()` loops |
| **Reusability** | Infinite (Can be read over and over) | Exhaustible (Can only be read through **once**) |

### **Creating generators**

- **🟢 Advantages of Generators :**
    * **Syntactic Simplicity:** They are cleaner and simpler to write than traditional list-generating functions, eliminating the boilerplate code of creating an empty list, calling `list.append()`, and returning the final collection in favor of a single `yield` statement.
    * **Minimal Memory Footprint:** Items are evaluated and streamed one at a time, removing the need to store massive datasets concurrently in system RAM.
    * **Dynamic Dependencies:** Values are computed dynamically at the exact moment they are requested, allowing results to change based on real-time external variables (such as checking a live queue or stack).
    * **Lazy Evaluation:** Execution is entirely lazy; if your application only requests the first few results, the remaining elements are never calculated. Between these sequential requests, the generator's state is completely frozen.

---

- **🔴 Disadvantages of Generators :**
    * **Single-Use Exhaustion:** Results can only be iterated through exactly once. After a generator yields its final item, it is completely depleted and cannot be reset or reused.
    * **Unknown/Infinite Size:** The absolute size of a generator is hidden until processing concludes. Because a generator can theoretically be infinite, attempting to forcefully cast it into a standard list (`list(infinite_generator)`) can instantly exhaust system memory and crash the Python interpreter.
    * **No Sequence Slicing:** Because items are generated on-demand rather than indexed sequentially in memory, slicing syntax like `generator[10:20]` is completely unsupported (though it can be bypassed using `itertools.islice`, which discards skipped elements).
    * **No Direct Indexing:** Arbitrary random access is impossible; you cannot instantly pull a specific element from a generator using bracket notation like `generator[5]`.

In [1]:
def generator():
    yield 1
    yield 'a'
    yield []
    return 'result'

In [2]:
result = generator()

result

<generator object generator at 0x7fd1ff783d70>

In [3]:
list(result)

[1, 'a', []]

In [4]:
list(result)

[]

In [6]:
def generator_with_return():
    yield "some value"
    return "exit generator function"

result = generator_with_return()

In [8]:
next(result)

'some value'

In [9]:
def lazy():
    print("Before yielding")
    yield "yielding"
    print("After yielding")

In [10]:
generator = lazy()

next(generator)

Before yielding


'yielding'

In [13]:
generator = lazy()

next(generator)

Before yielding


'yielding'

In [14]:
try:
    next(generator)
except StopIteration:
    pass

After yielding


In [16]:
for item in lazy():
    print(item)

Before yielding
yielding
After yielding


### **Creating infinite generators**

In [18]:
def count(start=0, step=1, stop=None):
    n = start
    while stop is not None and n < stop:
        yield n
        n += step

In [19]:
list(count(10, 2.5, 20))

[10, 12.5, 15.0, 17.5]

> Due to the potentially infinite nature of generators, caution is required. Without the stop variable, simply doing **list(count())** would result in an infinite loop that results in an out-of-memory situation quite fast.

### **Generators wrapping iterables**

In [20]:
def square(iterable):
    for i in iterable:
        yield i**2

In [21]:
list(square(range(5)))

[0, 1, 4, 9, 16]

In [22]:
def square(iterable):
    yield 'Begin'
    for i in iterable:
        yield i**2
    yield 'End'

In [23]:
list(square(range(5)))

['Begin', 0, 1, 4, 9, 16, 'End']

In [24]:
def odd(iterable):
    for i in iterable:
        if i % 2:
            yield i

def square(iterable):
    for i in iterable:
        yield i**2 

In [25]:
list(square(odd(range(10))))

[1, 9, 25, 49, 81]

### **Generator comprehensions**

In [26]:
squares = (x**2 for x in range(5))

squares

<generator object <genexpr> at 0x7fd1fd482740>

In [27]:
list(squares)

[0, 1, 4, 9, 16]

In [28]:
import itertools

In [33]:
result = itertools.count()

odd = (x for x in result if x % 2)
sliced_odd = itertools.islice(odd, 5)

list(sliced_odd)

[1, 3, 5, 7, 9]

In [34]:
result = itertools.count()
sliced_result = itertools.islice(result, 5)
odd_result = (x for x in sliced_result if x % 2)

list(odd_result)

[1, 3]

### **Class-based generators and iterators**

In [5]:
class CounterGenerator:
    def __init__(self, start=0, step=1, stop=None):
        self.start = start
        self.step = step
        self.stop = stop

    def __iter__(self):
        i = self.start
        while self.stop is None or i < self.stop:
            yield i
            i += self.step

In [7]:
list(CounterGenerator(10, 2.5, 20))

[10, 12.5, 15.0, 17.5]

In [8]:
class CountIterator:
    def __init__(self, start=0, step=1, stop=None):
        self.i = start
        self.start = start
        self.step = step
        self.stop = stop

    def __iter__(self):
        return self

    def __next__(self):
        if self.stop is not None and self.i >= self.stop:
            raise StopIteration

        value = self.i
        self.i += self.step
        return value

In [9]:
list(CountIterator(10, 2.5, 20))

[10, 12.5, 15.0, 17.5]

In [10]:
import itertools

In [14]:
class AdvancedCountIterator:

    def __init__(self, start=0, step=1, stop=None):
        self.start = start
        self.step = step
        self.stop = stop

    def __iter__(self):
        return self

    def __next__(self):
        if self.stop is not None and self.start >= self.stop:
            raise StopIteration

        value = self.start
        self.start += self.step
        return value

    def __len__(self):
        return int((self.stop - self.start) // self.step)

    def __contains__(self, key):
        return self.start < key < self.stop

    def __repr__(self):
        return (
            f"{self.__class__.__name__}(start={self.start}, "
            f"step={self.step}, stop={self.stop})"
        )

    def __getitem__(self, slice_):
        return itertools.islice(self, slice_.start, slice_.stop, slice_.step)

In [15]:
count = AdvancedCountIterator(10, 2.5, 20)

In [16]:
count

AdvancedCountIterator(start=10, step=2.5, stop=20)

In [17]:
3 in count

False

In [18]:
11 in count

True

In [19]:
11.3 in count

True

In [20]:
len(count)

4

In [21]:
list(count[:2])

[10, 12.5]

In [22]:
list(count)

[15.0, 17.5]

In [23]:
list(count)

[]

## **Generator examples**

> These generators work on **all iterables**, not just generators. So, you could also apply them to a **list**, **tuple**, **string**, or other kinds of **iterables**.

### **Breaking an iterable up into chunks/groups**

In [34]:
import itertools

In [35]:
def groups_of_chunks(iterable, n, fill_value=None):
    args = [iter(iterable)] * n
    return itertools.zip_longest(*args, fillvalue=fill_value)

In [36]:
list(groups_of_chunks("ABCDRFT", 3, 'M'))

[('A', 'B', 'C'), ('D', 'R', 'F'), ('T', 'M', 'M')]

In [40]:
def chunker(iterable, chunk_size):
    # Make sure 'iterable' is an iterator
    iterable = iter(iterable)

    def chunk(value):
        # Make sure not to skip the given value
        yield value
        # We already yielded a value so reduce the chunk_size
        for _ in range(chunk_size - 1):
            try:
                yield next(iterable)
            except StopIteration:
                break

    while True:
        try:
            # Check if we're at the end by using 'next()'
            yield chunk(next(iterable))
        except StopIteration:
            break

In [41]:
for chunk in chunker("ABfstdjk", 3):
    for value in chunk:
        print(value, end=', ')
    print()

A, B, f, 
s, t, d, 
j, k, 


### **itertools.islice – Slicing iterables**

In [42]:
some_list = list(range(1000))

some_list[:5]

[0, 1, 2, 3, 4]

In [44]:
list(itertools.islice(some_list, 5))

[0, 1, 2, 3, 4]

In [45]:
some_list[10:20:2]

[10, 12, 14, 16, 18]

In [46]:
list(itertools.islice(some_list, 10, 20, 2))

[10, 12, 14, 16, 18]

- While regular slicing and **`itertools.islice()`** can produce identical outputs, they are completely different under the hood regarding internal mechanics and computational efficiency. Regular slicing **(`sequence[start:stop:step]`)** requires the underlying object to natively support the slice protocol by implementing the **`__getitem__()`** magic method. For built-in sequences like lists or tuples, this operation is highly optimized and executes with a time complexity of $O(k)$ (where $k$ is the number of elements in the slice); it takes the exact same amount of time to grab 10 elements from the beginning of a list as it does from the very end because Python can jump directly to those memory addresses.

- Conversely, **`itertools.islice()`** makes no assumptions about indexing and only requires the input to be a basic iterable (like a generator). Because it cannot jump across memory, fetching elements late in a sequence requires it to sequentially iterate through and throw away every single preceding item. Consequently, slicing a list from index 900 to 910 takes a mere 10 steps, whereas doing the same with **`itertools.islice()`** forces Python to process 910 steps, resulting in a significantly heavier performance penalty for deep slices.

In [49]:
def islice(iterable, start, stop=None, step=1):
    # 'islice' has signatures: 'islice(iterable, stop)' and:
    # 'islice(iterable, start, stop[, step])'
    # 'fill' stop with 'start' if needed
    if stop is None and start is not None and step == 1:
        start, stop = 0, start

    # Create an iterator and discard the first 'start' items
    iterator = iter(iterable)
    for _ in range(start):
        next(iterator)

    # Enumerate the iterator making 'i' start at 'start'
    for i, item in enumerate(iterator, start):
        # Stop when we've reached 'stop' items
        if i >= stop:
            return
        # Use modulo 'step' to discard non-matching items
        if i % step:
            continue

        yield item

In [50]:
list(islice(range(1000), 10))

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [51]:
list(islice(range(1000), 900, 920, 2))

[900, 902, 904, 906, 908, 910, 912, 914, 916, 918]

### **itertools.chain – Concatenating multiple iterables**

In [52]:
def chain(*iterables):
    for iterable in iterables:
        yield from iterable

In [54]:
a = ["a", "b", "c"]
b = "a", "b", "c"
c = "abc"

list(chain(a, b, c))

['a', 'b', 'c', 'a', 'b', 'c', 'a', 'b', 'c']

In [55]:
def chain(*iterables):
    for iterable in iterables:
        for i in iterable:
            yield i

In [56]:
list(chain(a, b, c))

['a', 'b', 'c', 'a', 'b', 'c', 'a', 'b', 'c']

### **itertools.tee – Using an output multiple times**

In [57]:
import itertools

In [58]:
def spam_eggs():
    yield 'spam'
    yield 'eggs'

In [59]:
a, b = itertools.tee(spam_eggs())

In [60]:
next(a)

'spam'

In [61]:
next(a)

'eggs'

In [62]:
next(b)

'spam'

In [63]:
next(b)

'eggs'

In [64]:
next(b)

StopIteration: 

### **contextlib.contextmanager – Creating context managers**

In [65]:
import time
import datetime
import contextlib

In [67]:
@contextlib.contextmanager
def timer(name):
    start = datetime.datetime.now()
    yield
    stop = datetime.datetime.now()
    print("%s took %s" %(name, stop - start))

In [68]:
with timer('basic timer'):
    time.sleep(0.3)

basic timer took 0:00:00.300429


In [70]:
# Write standard print output to a file temporarily
@contextlib.contextmanager
def write_to_log(name):
    with open(f'{name}.txt', 'w') as fh:
        with contextlib.redirect_stdout(fh):
            with timer(name):
                yield

In [72]:
# Using as a decorator also works in addition to with-statements
@write_to_log('some_name')
def some_function():
    print("This will be written to 'some_name.txt'")

In [73]:
some_function()

In [74]:
@contextlib.contextmanager
def write_to_log(name):
    with contextlib.ExitStack() as stack:
        fh = stack.enter_context(open(f"{name}.txt", 'w'))
        stack.enter_context(contextlib.redirect_stdout(fh))
        stack.enter_context(timer(name))
        yield

In [75]:
@write_to_log('some_name1')
def some_function():
    print("This will be written to 'some_name1.txt'")

In [76]:
some_function()

In [77]:
with contextlib.ExitStack() as stack:
    fh = stack.enter_context(open('testx.txt', 'w'))
    # Move the context(s) to a new ExitStack
    new_stack = stack.pop_all()

In [78]:
bytes_written = fh.write('fh is still open')

In [79]:
# After closing we can't write anymore
new_stack.close()

In [80]:
fh.write('cant write anymore')

ValueError: I/O operation on closed file.

## **Coroutines**

A **coroutine** is an advanced evolution of a generator designed for high-performance **asynchronous programming and cooperative multitasking**.

While regular generators are pulling mechanisms (producing data via `yield`), coroutines are primarily data consumers or data processors. They allow execution to be paused and resumed dynamically, enabling Python to switch tasks and handle massive amounts of concurrent operations (like network requests or file inputs) without the heavy overhead of system threading.

---

- **🏎️ The Architectural Evolution: Generators vs. Coroutines**
    - The shift from a generator to a coroutine comes down to whether the data flows *out* or *in*:
        * **Generator (Producer):** Uses `yield value` to push data out to a loop.
        * **Coroutine (Consumer):** Uses `variable = yield` to pause execution and wait for data to be pushed *into* it from the outside.

---

- **🛠️ Classic Coroutines (The Native `yield` Method)**
    - Before Python introduced dedicated async keywords, coroutines were built natively using generators and the `.send()` method.

```python
def string_matcher(pattern):
    print(f"🎬 Coroutine initialized. Looking for: '{pattern}'")
    try:
        while True:
            # The coroutine pauses here, waiting for data to be sent in
            text = yield
            if pattern in text:
                print(f"🎯 Match Found: {text}")
    except GeneratorExit:
        print("🧹 Coroutine closed cleanly.")

# 1. Instantiate the coroutine
matcher = string_matcher("error")

# 2. Priming: Advance execution to the first yield statement
next(matcher)  # Output: 🎬 Coroutine initialized...

# 3. Stream data into the coroutine dynamically
matcher.send("System status: OK")      # (Ignored, doesn't contain 'error')
matcher.send("Warning: low memory")    # (Ignored)
matcher.send("Critical error detected") # Output: 🎯 Match Found: Critical error detected

# 4. Explicitly shut down the coroutine
matcher.close()  # Output: 🧹 Coroutine closed cleanly.
```

---

- **🚀 Modern Coroutines (`async` and `await`)**
    - In modern Python, the manual `yield`-based coroutine pattern is abstracted away into dedicated language syntax: **`async def`** and **`await`**. This framework powers high-concurrency engines like `asyncio`.
    - Instead of running linearly, an `async` coroutine yields control back to an underlying system event loop whenever it hits an `await` blocker (like a slow network download), allowing other code routines to execute in the meantime.

```python
import asyncio

async def fetch_api_data(endpoint, delay):
    print(f"📡 Requesting data from {endpoint}...")
    # Voluntarily releases control back to the event loop during this idle wait time
    await asyncio.sleep(delay) 
    print(f"✅ Received payload from {endpoint}")
    return {"status": 200, "endpoint": endpoint}

async def main():
    # Run multiple coroutines concurrently on a single system thread
    task1 = fetch_api_data("users_api", 2)
    task2 = fetch_api_data("orders_api", 1)
    
    # Gathers and runs them simultaneously
    results = await asyncio.gather(task1, task2)
    print("✨ All requests processed successfully.")

# Execute the event loop
asyncio.run(main())

# Output Order:
# 📡 Requesting data from users_api...
# 📡 Requesting data from orders_api...
# ✅ Received payload from orders_api   (Finishes first because delay was 1s)
# ✅ Received payload from users_api    (Finishes second)
# ✨ All requests processed successfully.
```

---

- **⚖️ Summary Comparison Matrix**

| Feature | Subroutines (Normal Functions) | Generators | Coroutines (`async/await`) |
| --- | --- | --- | --- |
| **Entry Points** | Single (Starts from the top line) | Single (Starts from the top line) | **Multiple** (Resumes exactly where it was paused) |
| **Data Interaction** | Receives inputs once at call time | Emits data out via `yield` | **Consumes and pauses** waiting for external data or events |
| **State Retention** | No (Destroyed upon reaching `return`) | Yes (Local variables are frozen during `yield`) | **Yes** (Maintains local variables and execution frame) |
| **Primary Use Case** | Basic sequential business logic | Memory-efficient data streaming | **High-concurrency IO** (Web servers, APIs, scrapers) |

### **A basic example**

In [81]:
def generator():
    value = yield 'value from generator'
    print("Generator recieved: ", value)
    yield f"Previous value: {value!r}"

In [82]:
g = generator()

print("print from generator: ", next(g))

print from generator:  value from generator


In [83]:
print(g.send("Value from Caller"))

Generator recieved:  Value from Caller
Previous value: 'Value from Caller'


### **Priming**

In [84]:
import functools

In [85]:
def coroutine(func):
    # Copy the 'function' description with 'functools.wraps'
    @functools.wraps(func)
    def _coroutine(*args, **kwargs):
        active_coroutine = func(*args, **kwargs)
        # Prime the coroutine and make sure we get no values
        assert not next(active_coroutine)
        return active_coroutine

    return _coroutine

In [86]:
@coroutine
def our_coroutine():
    while True:
        print("Waiting for yield...")
        value = yield
        print("our coroutine recieved: ", value)

In [87]:
generator = our_coroutine()

Waiting for yield...


In [88]:
generator.send("m")

our coroutine recieved:  m
Waiting for yield...


In [89]:
generator.send("b")

our coroutine recieved:  b
Waiting for yield...


In [91]:
next(generator)

our coroutine recieved:  None
Waiting for yield...


In [92]:
next(generator)

our coroutine recieved:  None
Waiting for yield...


> Note that the **coroutine decorator** will be used throughout this chapter from this point onward. For brevity, the coroutine function definition will be omitted from the following examples.

### **Closing and throwing exceptions**

In [93]:
@coroutine
def simple_coroutine():
    print('Setting up the coroutine')
    try:
        while True:
            item = yield
            print("Got Item: ", item)
    except GeneratorExit:
        print("Normal exit")
    except Exception as e:
        print("Exception exit: ", e)
        raise
    finally:
        print("Any exit")

In [94]:
active_coroutine = simple_coroutine()

Setting up the coroutine


In [95]:
active_coroutine.send("From caller")

Got Item:  From caller


In [97]:
active_coroutine.close()

Normal exit
Any exit


In [98]:
active_coroutine = simple_coroutine()

Setting up the coroutine


In [99]:
active_coroutine.throw(RuntimeError, "Caller sent an error")

Exception exit:  Caller sent an error
Any exit


/tmp/ipykernel_4519/4059301524.py:1: DeprecationWarning: the (type, exc, tb) signature of throw() is deprecated, use the single-arg signature instead.
  active_coroutine.throw(RuntimeError, "Caller sent an error")


RuntimeError: Caller sent an error

In [100]:
active_coroutine = simple_coroutine()

Setting up the coroutine


In [101]:
try:
    active_coroutine.throw(RuntimeError, "Caller sent an error")
except RuntimeError as exception:
    print("Exception: ", exception)

Exception exit:  Caller sent an error
Any exit
Exception:  Caller sent an error


/tmp/ipykernel_4519/3598173136.py:2: DeprecationWarning: the (type, exc, tb) signature of throw() is deprecated, use the single-arg signature instead.
  active_coroutine.throw(RuntimeError, "Caller sent an error")


### **Mixing generators and coroutines**

In [102]:
lines = 'some old text', 'really really old', 'old old old'

In [103]:
@coroutine
def replace(search, replace):
    while True:
        item = yield
        print(item.replace(search, replace))

In [104]:
old_replace = replace('old', 'new')

In [105]:
for line in lines:
    old_replace.send(line)

some new text
really really new
new new new


In [106]:
@coroutine
def replace(search, replace):
    while True:
        item = yield
        yield item.replace(search, replace)

In [107]:
old_replace = replace('old', 'new')

In [123]:
for line in lines:
    print(old_replace.send(line))

None
really really new
None


In [124]:
@coroutine
def replace(search, replace):
    item = yield
    while True:
        item = yield item.replace(search, replace)

In [125]:
old_replace = replace('old', 'new')

In [127]:
for line in lines:
    print(old_replace.send(line))

some new text
really really new
new new new


In [128]:
@coroutine
def replace(target, search, replace):
    while True:
        target.send((yield).replace(search, replace))

# Print will print the items using the provided formatstring
@coroutine
def print_(formatstring):
    count = 0
    while True: 
        count += 1
        print(count, formatstring.format((yield)))

# tee multiplexes the items to multiple targets
@coroutine
def tee(*targets):
    while True:
        item = yield
        for target in targets:
            target.send(item)

In [130]:
# Because we wrap the results we need to work backwards from the
# inner layer to the outer layer.

# First, create a printer for the items:
printer = print_("print: {}")

# Create replacers that send the output to the printer
old_replace = replace(printer, 'old', 'new')
current_replace = replace(printer, 'old', 'current')

# Send the input to both replacers
branch = tee(old_replace, current_replace)

# Send the data to the tee routine for processing
for line in lines:
    branch.send(line)

1 print: some new text
2 print: some current text
3 print: really really new
4 print: really really current
5 print: new new new
6 print: current current current
